# 03 — EDA criminal-temporal: preguntas 1–5

## Alcance

Este cuaderno responde únicamente las cinco primeras preguntas del
documento **EDA_Criminalidad_Cali.docx**. No contiene resultados
precalculados: las tablas, pruebas y gráficos se generan al ejecutarlo
en Colab con las salidas validadas de los notebooks 00–02.


## 1. Datos y comprobaciones previas


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

DATAV3_ROOT = Path("/content/drive/MyDrive/datav3")
SIEDCO_INPUT = DATAV3_ROOT / "A1 - SIEDCO" / "datos_criminalidad_cali"
PROJECT_OUTPUT = DATAV3_ROOT / "project_diplodata_outputs" / "eda_01_05_v1"
LANDING = PROJECT_OUTPUT / "landing"
TRUSTED = PROJECT_OUTPUT / "trusted"
SURFACE = PROJECT_OUTPUT / "surface"
AUDIT = PROJECT_OUTPUT / "audit"
REPORTS = PROJECT_OUTPUT / "reportes"

for directory in (LANDING, TRUSTED, SURFACE, AUDIT, REPORTS):
    directory.mkdir(parents=True, exist_ok=True)

PERIODO_INICIO = 2018
PERIODO_FIN = 2025

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")

eda_path = SURFACE / "analitica_eda.csv"
population_path = SURFACE / "poblacion_cali_anual.csv"
if not eda_path.is_file() or not population_path.is_file():
    raise FileNotFoundError("Ejecute los notebooks 00, 01 y 02 en ese orden.")

data = pd.read_csv(eda_path, low_memory=False)
population = pd.read_csv(population_path)
data["fecha"] = pd.to_datetime(data["fecha"], errors="coerce")
data["cantidad"] = pd.to_numeric(data["cantidad"], errors="coerce")
population["anio"] = pd.to_numeric(population["anio"], errors="raise").astype(int)
population["poblacion"] = pd.to_numeric(population["poblacion"], errors="raise")

analysis = data.loc[
    data["fecha"].dt.year.between(PERIODO_INICIO, PERIODO_FIN)
    & data["cantidad"].notna()
    & data["cantidad"].ge(0)
].copy()
analysis["anio"] = analysis["fecha"].dt.year.astype(int)

expected_crimes = {
    "HOMICIDIO", "HURTO A PERSONAS", "HURTO A RESIDENCIAS",
    "HURTO A COMERCIO", "HURTO DE MOTOCICLETAS", "HURTO DE AUTOMOTORES",
    "LESIONES PERSONALES", "VIOLENCIA INTRAFAMILIAR", "DELITOS SEXUALES",
    "AMENAZAS", "EXTORSION",
}
missing_crimes = sorted(expected_crimes - set(analysis["tipo_delito"].dropna()))
if missing_crimes:
    raise RuntimeError("Faltan categorías para las preguntas 1–5: " + ", ".join(missing_crimes))
if set(range(PERIODO_INICIO, PERIODO_FIN + 1)) - set(population["anio"]):
    raise RuntimeError("La población DANE no cubre todo el período 2018–2025.")

print(f"Filas agregadas usadas: {len(analysis):,}")
print("Recordatorio: los casos se calculan sumando cantidad, no contando filas.")


## Pregunta 1

**Pregunta:** ¿Cuál es la distribución de la cantidad total de delitos reportados en Cali por tipo de delito (homicidio, hurto a personas, hurto a residencias, hurto a comercio, hurto de motocicletas, hurto de automotores, lesiones personales, violencia intrafamiliar, delitos sexuales, amenazas y extorsión)?

**Código:** la resolución se desarrolla en la celda siguiente.

**Respuesta:** se entrega una tabla y un gráfico de barras ordenados. El total se calcula con `sum(cantidad)`, y la fuente complementaria solo aporta modalidades no solapadas que pudieron verificarse.

**Guía de interpretación:** diferencias de volumen no equivalen a riesgo individual ni corrigen subregistro.

**Interpretación o conclusión:** esta distribución sirve para priorizar el análisis descriptivo, no para concluir causalidad.

**Decisión que apoya:** seleccionar las categorías que requieren revisión prioritaria y justificar dónde profundizar el análisis, sin asignar recursos automáticamente.


In [ ]:
q1 = (
    analysis.groupby("tipo_delito", as_index=False)["cantidad"]
    .sum()
    .rename(columns={"cantidad": "total_reportado_2018_2025"})
    .sort_values("total_reportado_2018_2025", ascending=False)
)
q1["proporcion"] = (
    q1["total_reportado_2018_2025"]
    / q1["total_reportado_2018_2025"].sum()
)
display(q1)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=q1,
    y="tipo_delito",
    x="total_reportado_2018_2025",
    color="#2F5597",
)
plt.title("Cantidad total reportada por tipo de delito, Cali (2018–2025)")
plt.xlabel("Reportes agregados (suma de cantidad)")
plt.ylabel("Tipo de delito")
plt.tight_layout()
plt.savefig(REPORTS / "q1_distribucion_delitos.png", dpi=150, bbox_inches="tight")
plt.show()
q1.to_csv(REPORTS / "q1_distribucion_delitos.csv", index=False, encoding="utf-8-sig")


## Pregunta 2

**Pregunta:** ¿Cuál es la tasa de criminalidad por 100.000 habitantes para cada tipo de delito en Cali? ¿Cuáles son los delitos de mayor y menor incidencia?

**Código:** la resolución se desarrolla en la celda siguiente.

**Respuesta:** se calculan tasas anuales y un resumen comparable solo para categorías con los ocho años completos. La fórmula anual es `cantidad anual / población del mismo año × 100.000`, y el resumen del período usa personas-año: `sum(cantidad anual) / sum(población anual) × 100.000`.

**Guía de interpretación:** aquí "incidencia" se usa como tasa de reportes, no como incidencia epidemiológica ni como medida directa de victimización real.

**Interpretación o conclusión:** las categorías con cobertura temporal incompleta no deben ordenarse ni compararse como si tuvieran la misma base analítica.

**Decisión que apoya:** comparar la presión anual de los reportes con una base poblacional común y evitar priorizaciones sustentadas en denominadores incompatibles.


In [ ]:
annual = (
    analysis.groupby(["anio", "tipo_delito"], as_index=False)["cantidad"]
    .sum()
    .rename(columns={"cantidad": "cantidad_anual"})
)
q2_annual = annual.merge(
    population[["anio", "poblacion"]],
    on="anio",
    how="left",
    validate="many_to_one",
)
if q2_annual["poblacion"].isna().any() or (q2_annual["poblacion"] <= 0).any():
    raise RuntimeError("Falta población anual válida para calcular tasas.")
q2_annual["tasa_100k"] = (
    q2_annual["cantidad_anual"] / q2_annual["poblacion"] * 100_000
)

q2_period = (
    q2_annual.groupby("tipo_delito", as_index=False)
    .agg(
        cantidad_periodo=("cantidad_anual", "sum"),
        personas_anio=("poblacion", "sum"),
        anios_observados=("anio", "nunique"),
    )
)
q2_period["comparable_8_anios"] = q2_period["anios_observados"].eq(8)
q2_period["tasa_periodo_por_100k_personas_anio"] = np.where(
    q2_period["comparable_8_anios"],
    q2_period["cantidad_periodo"] / q2_period["personas_anio"] * 100_000,
    np.nan,
)
q2_period = q2_period.sort_values(
    "tasa_periodo_por_100k_personas_anio", ascending=False
)
display(q2_annual.sort_values(["tipo_delito", "anio"]))
display(q2_period)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=q2_period.dropna(subset=["tasa_periodo_por_100k_personas_anio"]),
    y="tipo_delito",
    x="tasa_periodo_por_100k_personas_anio",
    color="#4472C4",
)
plt.title("Tasa de reportes por tipo de delito, Cali (2018–2025)")
plt.xlabel("Reportes por 100.000 personas-año")
plt.ylabel("Tipo de delito")
plt.tight_layout()
plt.savefig(REPORTS / "q2_tasas_periodo.png", dpi=150, bbox_inches="tight")
plt.show()

comparable = q2_period.dropna(subset=["tasa_periodo_por_100k_personas_anio"])
if comparable.empty:
    raise RuntimeError("Ninguna categoría tiene ocho años comparables.")
display(
    pd.DataFrame(
        {
            "extremo": ["Mayor tasa del período", "Menor tasa del período"],
            "tipo_delito": [
                comparable.iloc[0]["tipo_delito"],
                comparable.iloc[-1]["tipo_delito"],
            ],
            "tasa": [
                comparable.iloc[0]["tasa_periodo_por_100k_personas_anio"],
                comparable.iloc[-1]["tasa_periodo_por_100k_personas_anio"],
            ],
        }
    )
)
q2_annual.to_csv(REPORTS / "q2_tasas_anuales.csv", index=False, encoding="utf-8-sig")
q2_period.to_csv(REPORTS / "q2_tasas_periodo.csv", index=False, encoding="utf-8-sig")


## Pregunta 3

**Pregunta:** ¿Cómo ha evolucionado la cantidad de cada tipo de delito en el período 2018-2025? ¿Existe una tendencia general creciente, estable o decreciente?

**Código:** la resolución se desarrolla en la celda siguiente.

**Respuesta:** se construye la serie anual por delito y se reportan pendiente lineal, valor p y valor q ajustado con Benjamini-Hochberg. Solo se clasifica una categoría cuando tiene los ocho años completos.

**Guía de interpretación:** `ESTABLE/NO CONCLUYENTE` significa que no se detectó una pendiente distinta de cero al 5%; no demuestra ausencia de cambios.

**Interpretación o conclusión:** la clasificación es descriptiva y sirve para priorizar revisión analítica, no para hacer pronósticos.

**Decisión que apoya:** identificar qué series ameritan revisión contextual o seguimiento reforzado antes de formular una intervención o un modelo.


In [ ]:
def benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    ranked = p[order]
    adjusted = ranked * len(p) / np.arange(1, len(p) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    result = np.empty_like(adjusted)
    result[order] = np.clip(adjusted, 0, 1)
    return result

trend_rows = []
for crime, group in annual.groupby("tipo_delito"):
    group = group.sort_values("anio")
    if group["anio"].nunique() != 8:
        trend_rows.append(
            {
                "tipo_delito": crime,
                "anios_observados": group["anio"].nunique(),
                "pendiente_anual": np.nan,
                "p_value": np.nan,
            }
        )
        continue
    fit = stats.linregress(group["anio"], group["cantidad_anual"])
    trend_rows.append(
        {
            "tipo_delito": crime,
            "anios_observados": 8,
            "pendiente_anual": fit.slope,
            "p_value": fit.pvalue,
            "r_cuadrado": fit.rvalue ** 2,
        }
    )

q3_trends = pd.DataFrame(trend_rows)
valid = q3_trends["p_value"].notna()
q3_trends.loc[valid, "q_value_bh"] = benjamini_hochberg(
    q3_trends.loc[valid, "p_value"]
)
q3_trends["tendencia"] = "COBERTURA INCOMPLETA"
q3_trends.loc[
    valid & (q3_trends["q_value_bh"] > 0.05), "tendencia"
] = "ESTABLE/NO CONCLUYENTE"
q3_trends.loc[
    valid
    & (q3_trends["q_value_bh"] <= 0.05)
    & (q3_trends["pendiente_anual"] > 0),
    "tendencia",
] = "CRECIENTE"
q3_trends.loc[
    valid
    & (q3_trends["q_value_bh"] <= 0.05)
    & (q3_trends["pendiente_anual"] < 0),
    "tendencia",
] = "DECRECIENTE"
display(q3_trends.sort_values(["tendencia", "tipo_delito"]))

plt.figure(figsize=(11, 7))
sns.lineplot(
    data=annual,
    x="anio",
    y="cantidad_anual",
    hue="tipo_delito",
    marker="o",
)
plt.title("Evolución anual por tipo de delito, Cali (2018–2025)")
plt.xlabel("Año")
plt.ylabel("Reportes agregados por año (suma de cantidad)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Tipo de delito")
plt.tight_layout()
plt.savefig(REPORTS / "q3_evolucion_anual.png", dpi=150, bbox_inches="tight")
plt.show()
q3_trends.to_csv(REPORTS / "q3_tendencias.csv", index=False, encoding="utf-8-sig")


## Pregunta 4

**Pregunta:** ¿Cuántos tipos de delito muestran tendencia creciente, estable o decreciente en el período analizado?

**Código:** la resolución se desarrolla en la celda siguiente.

**Respuesta:** se entrega un conteo por clasificación. Las coberturas incompletas se muestran aparte y no se fuerzan dentro de la categoría estable.

**Guía de interpretación:** los conteos dependen del umbral estadístico usado y de trabajar con solo ocho observaciones anuales.

**Interpretación o conclusión:** cada serie debe revisarse individualmente antes de pensar en intervención o modelado.

**Decisión que apoya:** ordenar la revisión de las categorías por tipo de señal y reconocer explícitamente cuáles no tienen evidencia concluyente.


In [ ]:
q4 = (
    q3_trends.groupby("tendencia", as_index=False)
    .size()
    .rename(columns={"size": "numero_tipos_delito"})
    .sort_values("numero_tipos_delito", ascending=False)
)
display(q4)
q4.to_csv(REPORTS / "q4_resumen_tendencias.csv", index=False, encoding="utf-8-sig")


## Pregunta 5

**Pregunta:** ¿En qué año fue máxima y mínima la cantidad de homicidios y hurtos a personas en Cali? ¿Existe algún patrón temporal común entre tipos de delito?

**Código:** la resolución se desarrolla en la celda siguiente.

**Respuesta:** se identifican los extremos para Homicidio y Hurto a Personas, y se estima la correlación de Spearman entre cambios anuales de `log1p(cantidad)`.

**Guía de interpretación:** correlación no implica causalidad. Con siete cambios anuales por serie, el resultado debe leerse como exploratorio.

**Interpretación o conclusión:** cualquier patrón aparente debe contrastarse con cambios de cobertura, definiciones y contexto antes de modelar.

**Decisión que apoya:** seleccionar períodos que requieren explicación contextual y decidir si existe fundamento para estudiar relaciones temporales con mayor detalle.


In [ ]:
focus_crimes = ["HOMICIDIO", "HURTO A PERSONAS"]
extreme_rows = []
for crime in focus_crimes:
    series = annual.loc[annual["tipo_delito"].eq(crime)].sort_values("anio")
    if series["anio"].nunique() != 8:
        raise RuntimeError(f"{crime} no tiene ocho años completos.")
    minimum = series.loc[series["cantidad_anual"].idxmin()]
    maximum = series.loc[series["cantidad_anual"].idxmax()]
    extreme_rows.extend(
        [
            {
                "tipo_delito": crime,
                "extremo": "MÍNIMO",
                "anio": int(minimum["anio"]),
                "cantidad": minimum["cantidad_anual"],
            },
            {
                "tipo_delito": crime,
                "extremo": "MÁXIMO",
                "anio": int(maximum["anio"]),
                "cantidad": maximum["cantidad_anual"],
            },
        ]
    )
q5_extremes = pd.DataFrame(extreme_rows)
display(q5_extremes)

annual_wide = annual.pivot(
    index="anio", columns="tipo_delito", values="cantidad_anual"
)
complete_crimes = annual_wide.columns[annual_wide.notna().all()].tolist()
annual_changes = np.log1p(annual_wide[complete_crimes]).diff().dropna(how="all")
q5_correlation = annual_changes.corr(method="spearman")
display(q5_correlation)

if q5_correlation.shape[0] >= 2:
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        q5_correlation,
        vmin=-1,
        vmax=1,
        center=0,
        cmap="vlag",
        square=True,
        cbar_kws={"label": "Coeficiente de Spearman (sin unidad)"},
    )
    plt.title("Correlación de cambios anuales entre tipos de delito")
    plt.tight_layout()
    plt.savefig(
        REPORTS / "q5_correlacion_cambios_anuales.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()

q5_extremes.to_csv(REPORTS / "q5_extremos.csv", index=False, encoding="utf-8-sig")
q5_correlation.to_csv(
    REPORTS / "q5_correlacion_cambios.csv", encoding="utf-8-sig"
)


## Cierre y límites

Los resultados válidos son exclusivamente los producidos por esta
ejecución. Antes de citarlos, revisar los inventarios, las banderas de
calidad y la decisión de solapamiento. Este notebook no responde las
preguntas 6–55.

## Producto diferencial

Estas cinco preguntas forman una vigilancia temporal auditable para Cali.
No estiman riesgo por comuna o barrio ni generan decisiones operativas
automáticas. Su función es aportar evidencia trazable para seguimiento,
planeación analítica y evaluación responsable.
